# Data cleaning for PPSV 2022 data

- Pedal poll dates 2021: June 1-6, 2021
- Pedal poll dates 2022: June 7-12 2022

"At each official count location, the goal was for 120-minute long counts during four weekday periods (07–09:00, 11–13:00, 14–16:00 and 16–18:00) and one weekend period (12–14:00)."

Data were downloaded from the Counterpoint Data Explorer (now replaced by superset.counterpointapp.org) for June 2022.

Cleaning / amalgamation steps (same steps as 2021 analysis):

- [x] remove counts outside Canada (downloaded Canada only)
- [x] remove sessions lasting less than 5 minutes (as in 2021)
- [x] remove sessions lasting more than 4 hours (as in 2021)
- [x] replace counts with values other than 1 or NULL in demographic columns within unknown - gender, age and ethnicity entries with values from 2 to 5
- [x] create demographic summary column
- [x] check for duplicated sessions and remove (happened in 2021)
- [x] remove sessions outside the count window unless they are between 1 and 3 hours and take place during one of the designated count time windows or have the "Pedal Poll Canada" team name
- [x] add paper count data
- [x] remove duplicated app and paper count data

In [1]:
import pandas as pd
import numpy as np

In [2]:
app_data_2022 = pd.read_csv("data/CounterPoint Data Explorer - Bicycle Demographics Submission Data_Page 1_Table.csv")
rows_removed = 0 # initialize counter to test later
original_shape = app_data_2022.shape[0]

In [3]:
hand_count_2022 = pd.read_csv("data/2022_hand_count.csv", low_memory=False)

In [4]:
# remove blank rows
hand_count_2022 = hand_count_2022[~np.all(hand_count_2022.isna(), axis = 1)]

In [5]:
# create file to assign counterpoint IDs
hand_count_2022.drop_duplicates(subset = 'counterpoint_name')[['counterpoint_id',
                                                               'counterpoint_name', 'lat',
                                                               'lng', 'geocoded_city',
                                                               'province']].to_csv("data/hand_count_2022_unique_counterpoints.csv", index = None)

Missing counterpoint IDs from the paper count were identified by hand where possible. When multiple nearby app counterpoint locations existed, one was chosen from the group. Some locations were not able to be assigned to a counterpoint ID; these are excluded from analysis.

In [6]:
# load filled counterpoint IDs and merge
counterpoint_ids_2022 = pd.read_csv("data/hand_count_2022_unique_counterpoints_filled.csv")

hand_count_2022 = hand_count_2022.drop(columns = 'counterpoint_id').merge(counterpoint_ids_2022[['counterpoint_id', 
                                                                        'counterpoint_name']], on = 'counterpoint_name')

In [7]:
# set column names to match 2021 columns
app_data_2022.columns = ['Submission Id', 'survey_session_id', 'start_datetime', 'end_datetime',
       'counterpoint_id', 'counterpoint_name', 'team_name', 'date', 'Lat/Lng',
       'geocoded_city', 'geocoded_province', 'geocoded_country', 'gender_woman', 'gender_man', 'gender_unsure',
        'age_child', 'age_adult', 'age_senior',
       'age_unsure',
       'race_black', 'race_east_asian', 'race_indigenous',
       'race_latin', 'race_middle_eastern', 'race_south_asian', 'race_white',
       'race_other_unsure']

In [8]:
# make separate lat and lng columns
app_data_2022[['lat', 'lng']] = app_data_2022['Lat/Lng'].str.split(',', expand = True)

In [9]:
app_data_2022['elapsed_time_seconds'] = (pd.to_datetime(app_data_2022['end_datetime']) 
                                         - pd.to_datetime(app_data_2022['start_datetime'])).dt.seconds

In [10]:
short_counts = app_data_2022[app_data_2022['elapsed_time_seconds'] < 5*60].shape[0]
print("We removed sessions shorter than 5 minutes (%s counts)" %short_counts)

We removed sessions shorter than 5 minutes (394 counts)


In [11]:
long_counts = app_data_2022[app_data_2022['elapsed_time_seconds'] > 4*60*60].shape[0]
print("We removed sessions longer than 4 hours (%s counts)" %long_counts)

We removed sessions longer than 4 hours (0 counts)


In [12]:
# remove short sessions
app_data_2022 = app_data_2022[app_data_2022['elapsed_time_seconds'] >= 5*60]
rows_removed += short_counts
rows_removed += long_counts

Replace demographic data with values other than one or NULL with unknown

- create summary demographic columns
- check for values other than 0 or 1 or multiple categories assigned as 1
- replace all three demographic values with 'Other or unsure' (as in 2021 data)
- list 'error' in 'error' column, 'no error' otherwise

In [13]:
# includes both hand and app count
data_2021_url = "https://velocanadabikes.org/wp-content/uploads/2022/Pedal%20Poll%202021%20Data%20UPLOAD.csv"
data_2021 = pd.read_csv(data_2021_url, index_col = 0)

In [14]:
data_2021.columns

Index(['counterpoint_id', 'counterpoint_name', 'lat', 'lng', 'coordinates',
       'geocoded_city', 'geocoded_province', 'user_id', 'survey_session_id',
       'elapsed_time_minutes', 'start_datetime', 'end_datetime', 'date',
       'start_datetime_year', 'start_datetime_month',
       'start_datetime_dayofmonth', 'start_datetime_dayofweek',
       'start_datetime_hourofday', 'timezone', 'gender_man', 'gender_woman',
       'gender_other_unsure', 'age_child', 'age_adult', 'age_senior',
       'age_unsure', 'race_black', 'race_east_asian', 'race_indigenous',
       'race_latin', 'race_middle_eastern', 'race_south_asian', 'race_white',
       'race_other_unsure', 'Name', 'gender', 'age', 'race', 'error'],
      dtype='object')

Check the values used in the 2021 data

In [15]:
data_2021[~np.all(data_2021[['gender_man', 'gender_woman',
       'gender_other_unsure', 'age_child', 'age_adult', 'age_senior',
       'age_unsure', 'race_black', 'race_east_asian', 'race_indigenous',
       'race_latin', 'race_middle_eastern', 'race_south_asian', 'race_white',
       'race_other_unsure']].isin([0,1]), axis = 1)][['gender', 'age', 'race', 'error']].value_counts()

gender           age     race             error
Other or unsure  Unsure  Other or unsure  error    437
Name: count, dtype: int64

In [16]:
data_2021['error'].unique()

array(['no error', 'error'], dtype=object)

In [17]:
data_2021['gender'].unique()

array(['Man', 'Woman', 'Other or unsure'], dtype=object)

In [18]:
data_2021['race'].unique()

array(['White', 'Other or unsure', 'South Asian', 'Indigenous',
       'East Asian', 'Black', 'Middle Eastern', 'Latin'], dtype=object)

In [19]:
data_2021['age'].unique()

array(['Unsure', 'Senior', 'Adult', 'Child'], dtype=object)

In [20]:
app_data_2022[(np.sum(app_data_2022[['age_child', 'age_adult', 'age_senior', 'age_unsure']], axis = 1) > 1)].shape

(59, 30)

In [21]:
# get error entries
# these are cases when there is more than one box ticked in a category
error_entries = app_data_2022[(np.sum(app_data_2022[['gender_woman', 'gender_man', 'gender_unsure']], axis = 1) > 1)
             | (np.sum(app_data_2022[['age_child', 'age_adult', 'age_senior', 'age_unsure']], axis = 1) > 1)
             | (np.sum(app_data_2022[['race_black', 'race_east_asian', 'race_indigenous', 'race_latin',
       'race_middle_eastern', 'race_south_asian', 'race_white', 'race_other_unsure']], axis = 1) > 1)]

print(error_entries.shape)

(110, 30)


In [22]:
num_error_gender = app_data_2022[(np.sum(app_data_2022[['gender_woman', 'gender_man', 'gender_unsure']], axis = 1) > 1)].shape[0]
num_error_age = app_data_2022[(np.sum(app_data_2022[['age_child', 'age_adult', 'age_senior', 'age_unsure']], axis = 1) > 1)].shape[0]
num_error_race = app_data_2022[(np.sum(app_data_2022[['race_black', 'race_east_asian', 'race_indigenous', 'race_latin',
       'race_middle_eastern', 'race_south_asian', 'race_white', 'race_other_unsure']], axis = 1) > 1)].shape[0]

In [23]:
print("%s bicyclists changed to 'Other or unsure': %s gender, %s age, %s race" %(error_entries.shape[0], num_error_gender,
                                                                                 num_error_age, num_error_race))

110 bicyclists changed to 'Other or unsure': 39 gender, 59 age, 59 race


Create summary demographic columns

In [24]:
# set error flag for app error entries
app_data_2022['error'] = 'no error' # initialize column
app_data_2022.loc[error_entries.index, 'error'] = 'error'

In [25]:
# check that something happened
app_data_2022['error'].unique()

array(['no error', 'error'], dtype=object)

In [26]:
# possible values to put in the 'gender' column
gender_vals = ['Woman', 'Man', 'Other or unsure']

In [27]:
# confirm that there are no demographics with all zeros, it has to be at least 1
# if there were all zeros, the next step would just take the 1st column which we DON'T want
assert np.any(np.sum(app_data_2022[['gender_woman', 'gender_man', 'gender_unsure']], axis = 1) == 0) == False
assert np.any(np.sum(app_data_2022[['age_child', 'age_adult', 'age_senior', 'age_unsure']], axis = 1) == 0) == False
assert np.any(np.sum(app_data_2022[['race_black', 'race_east_asian', 'race_indigenous', 'race_latin',
       'race_middle_eastern', 'race_south_asian', 'race_white', 'race_other_unsure']], axis = 1) == 0) == False

In [28]:
# set demographic columns to the max value among the options
app_data_2022['gender_column'] = app_data_2022[['gender_woman', 'gender_man', 'gender_unsure']].idxmax(1)
app_data_2022['age_column'] = app_data_2022[['age_child', 'age_adult', 'age_senior', 'age_unsure']].idxmax(1)
app_data_2022['race_column'] = app_data_2022[['race_black', 'race_east_asian', 'race_indigenous', 'race_latin',
       'race_middle_eastern', 'race_south_asian', 'race_white', 'race_other_unsure']].idxmax(1)

In [29]:
# map the column names to the intended final values
gender_vals_dict = {'gender_woman':'Woman', 'gender_man':'Man', 'gender_unsure':'Other or unsure'}
age_vals_dict = {'age_child':'Child', 'age_adult':'Adult', 'age_senior':'Senior', 'age_unsure':'Other or unsure'}
race_vals_dict = {'race_black':'Black', 'race_east_asian':'East Asian', 'race_indigenous':'Indigenous', 'race_latin':'Latin',
       'race_middle_eastern':'Middle Eastern', 'race_south_asian':'South Asian', 'race_white':'White', 'race_other_unsure':'Other or unsure'}

app_data_2022['gender'] =  app_data_2022['gender_column'].map(gender_vals_dict)
app_data_2022['age'] =  app_data_2022['age_column'].map(age_vals_dict)
app_data_2022['race'] =  app_data_2022['race_column'].map(race_vals_dict)

In [30]:
# as in 2021, set all fields to other or unsure for error columns
app_data_2022.loc[app_data_2022['error'] == 'error', 'gender'] = 'Other or unsure'
app_data_2022.loc[app_data_2022['error'] == 'error', 'age'] = 'Other or unsure'
app_data_2022.loc[app_data_2022['error'] == 'error', 'race'] = 'Other or unsure'

In [31]:
# check that it looks right
app_data_2022[app_data_2022['error'] == 'error'][['gender_column', 'age_column', 'race_column', 'gender', 'age', 'race']]

,gender_column,age_column,race_column,gender,age,race
6,gender_woman,age_adult,race_indigenous,Other or unsure,Other or unsure,Other or unsure
226,gender_woman,age_adult,race_white,Other or unsure,Other or unsure,Other or unsure
3194,gender_man,age_adult,race_south_asian,Other or unsure,Other or unsure,Other or unsure
3451,gender_woman,age_adult,race_white,Other or unsure,Other or unsure,Other or unsure
4667,gender_man,age_adult,race_latin,Other or unsure,Other or unsure,Other or unsure
...,...,...,...,...,...,...
47698,gender_woman,age_adult,race_indigenous,Other or unsure,Other or unsure,Other or unsure
47699,gender_man,age_adult,race_indigenous,Other or unsure,Other or unsure,Other or unsure
47700,gender_woman,age_adult,race_other_unsure,Other or unsure,Other or unsure,Other or unsure
47701,gender_man,age_adult,race_indigenous,Other or unsure,Other or unsure,Other or unsure


### Check for duplicated survey sessions as in 2021

In [32]:
# add column to compare totals across years - every row is a single count
app_data_2022['easy_bike_total'] = 1

In [33]:
app_data_2022['start_datetime_hourofday'] = pd.to_datetime(app_data_2022['start_datetime']).dt.hour
app_data_2022['start_datetime_dayofmonth'] = pd.to_datetime(app_data_2022['start_datetime']).dt.day
app_data_2022['start_datetime_dayofweek'] = pd.to_datetime(app_data_2022['start_datetime']).dt.day_name()

In [34]:
app_data_2022.groupby(['counterpoint_id', 'start_datetime_hourofday', 'start_datetime_dayofmonth',
                       'survey_session_id'])[['easy_bike_total', 'race_white']].sum().reset_index().shape

(695, 6)

In [35]:
app_data_2022.groupby(['counterpoint_id', 'start_datetime_hourofday', 
                       'start_datetime_dayofmonth'])[['easy_bike_total', 'race_white']].sum().reset_index().shape

(588, 5)

There are over 100 counts with the same location and start time and day but different survey session ID - check total bikes counted to see if these are duplicate counts.

In [36]:
sessions = app_data_2022.groupby(['counterpoint_id', 'start_datetime_hourofday', 'start_datetime_dayofmonth',
                       'survey_session_id'])['easy_bike_total'].sum().reset_index()

In [37]:
sessions.shape

(695, 5)

In [38]:
possible_duplicates = sessions[['counterpoint_id', 'start_datetime_hourofday', 
                       'start_datetime_dayofmonth', 'easy_bike_total']].value_counts().reset_index()

In [39]:
possible_duplicates = possible_duplicates[possible_duplicates['count'] > 1]

Another reason this can happen is someone stopping and starting the app and just coincidentally counting the same number of bikes. Check if the start and end times are consistent with this

In [40]:
for i, row in possible_duplicates.iterrows():
    group = app_data_2022[(app_data_2022['counterpoint_id'] == row['counterpoint_id'])
              & (app_data_2022['start_datetime_hourofday'] == row['start_datetime_hourofday'])
              & (app_data_2022['start_datetime_dayofmonth'] == row['start_datetime_dayofmonth'])]

    g2 = group[['survey_session_id', 'start_datetime', 'end_datetime']].drop_duplicates()
    
    for i in range(len(g2)-1):
        try:
            # check if end time is less than next start time
            assert g2.sort_values(by = 'start_datetime').iloc[i]['end_datetime'] < g2.sort_values(by = 'start_datetime').iloc[i+1]['start_datetime']
        except: # these ones don't seem genuine
            print(row['counterpoint_id'])

6412


For this case, the start and end times and everything else are identical, just a different survey session ID.

In [41]:
duplicated = app_data_2022[(app_data_2022['counterpoint_id'] == 6412)
              & (app_data_2022['start_datetime_hourofday'] == 15)]

The totals are not quite the same in each category either, though the total bikes are the same. How would this happen in the app? Take the one with the latest date, since this may reflect the person editing then saving their data. For our purposes we just use total bikes anyway.

In [42]:
duplicated.groupby(['survey_session_id', 'date','start_datetime', 'end_datetime', 'elapsed_time_seconds', 'counterpoint_id', 'geocoded_city'])[['gender_woman', 'gender_man',
       'gender_unsure', 'age_child', 'age_adult', 'age_senior', 'age_unsure',
       'race_black', 'race_east_asian', 'race_indigenous', 'race_latin',
       'race_middle_eastern', 'race_south_asian', 'race_white',
       'race_other_unsure']].sum()

gender_woman  \
survey_session_id date                 start_datetime   end_datetime     elapsed_time_seconds counterpoint_id geocoded_city                 
3623              Jun 8, 2022, 3:53 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 0   
                  Jun 8, 2022, 3:54 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 1   
                  Jun 8, 2022, 3:55 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 0   
                  Jun 8, 2022, 3:58 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 2   
                  Jun 8, 2022, 3:59 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 0   
...                                                                                                                                   ...   
3624              Jun 8, 2022, 5:44 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 1   
                  Jun 8, 2022, 5:47 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 2   
                  Jun 8, 2022, 5:49 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 1   
                  Jun 8, 2022, 5:50 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 1   
                  Jun 8, 2022, 5:52 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                 1   

                                                                                                                             gender_man  \
survey_session_id date                 start_datetime   end_datetime     elapsed_time_seconds counterpoint_id geocoded_city               
3623              Jun 8, 2022, 3:53 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               1   
                  Jun 8, 2022, 3:54 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               0   
                  Jun 8, 2022, 3:55 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               0   
                  Jun 8, 2022, 3:58 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               1   
                  Jun 8, 2022, 3:59 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               3   
...                                                                                                                                 ...   
3624              Jun 8, 2022, 5:44 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               0   
                  Jun 8, 2022, 5:47 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               0   
                  Jun 8, 2022, 5:49 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               1   
                  Jun 8, 2022, 5:50 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               1   
                  Jun 8, 2022, 5:52 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn               2   

                                                                                                                             gender_unsure  \
survey_session_id date                 start_datetime   end_datetime     elapsed_time_seconds counterpoint_id geocoded_city                  
3623              Jun 8, 2022, 3:53 PM 2022-06-08 15:52 2022-06-08 17:53 7260                 6412            Goulbourn                  0   
                  Jun 8, 2022, 3:54 PM 2022-06-08 15:52 2

In [43]:
rows_removed_duplicate = app_data_2022[app_data_2022['survey_session_id'] == 3623].shape[0]
print(rows_removed_duplicate) # this is how many bikes are removed
app_data_2022 = app_data_2022[app_data_2022['survey_session_id'] != 3623]

144


### Keep data from outside the count week only if it has the Pedal Poll Canada team name

In [44]:
start_date = 7
end_date = 12

In [45]:
after_week_data = app_data_2022[app_data_2022['start_datetime_dayofmonth'] > end_date][['start_datetime_dayofmonth','team_name']]


In [46]:
before_week_data = app_data_2022[app_data_2022['start_datetime_dayofmonth'] < start_date][['start_datetime_dayofmonth','team_name']]

In [47]:
# this captures if someone starts their 7 am count at 6:59, for instance
valid_weekday_start_hours = [6,7, 10,11, 13,14, 15,16]
valid_weekend_start_hours = [11,12]
threshold_from_2_hours = 60 # minutes

In [48]:
outside_data = app_data_2022[(app_data_2022['start_datetime_dayofmonth'] > end_date)
                            | (app_data_2022['start_datetime_dayofmonth'] < start_date)]

In [49]:
valid_counts = outside_data[
            # weekend data
            (((outside_data['start_datetime_dayofweek'] == 'Saturday')
              | (outside_data['start_datetime_dayofweek'] == 'Sunday'))
             & (outside_data['start_datetime_hourofday'].isin(valid_weekend_start_hours)))
            # weekday data
           | (((outside_data['start_datetime_dayofweek'] != 'Saturday')
              & (outside_data['start_datetime_dayofweek'] != 'Sunday'))
             & (outside_data['start_datetime_hourofday'].isin(valid_weekday_start_hours)))
]

In [50]:
threshold_from_2_hours = 60

In [51]:
valid_counts = valid_counts[np.abs((valid_counts['elapsed_time_seconds'] /60) - 120 )  <= threshold_from_2_hours]

In [52]:
# this looks ok
valid_counts.drop_duplicates(subset = 'survey_session_id')[['elapsed_time_seconds', 'start_datetime_dayofmonth', 
                                                              'start_datetime_hourofday', 'start_datetime_dayofweek', 
                                                              'team_name']]

,elapsed_time_seconds,start_datetime_dayofmonth,start_datetime_hourofday,start_datetime_dayofweek,team_name
18,4140,23,7,Thursday,No team
48972,5640,6,14,Monday,Pedal Poll Canada
48977,7140,6,15,Monday,No team
49266,7320,6,6,Monday,Kingston Velo


In [53]:
primary_data = app_data_2022[(app_data_2022['start_datetime_dayofmonth'] <= end_date)
                            | (app_data_2022['start_datetime_dayofmonth'] >= start_date)]

In [54]:
rows_removed = rows_removed + app_data_2022.shape[0] - primary_data.shape[0]

In [55]:
app_data_2022 = pd.concat([primary_data, valid_counts])

In [56]:
rows_removed -= valid_counts.shape[0]

In [57]:
valid_counts.shape[0]

100

In [58]:
# drop temporary columns and columns that aren't in 2021 data
app_data_2022 = app_data_2022.drop(columns = ['gender_column', 'age_column',
       'race_column', 'Submission Id', 'geocoded_country', 'Lat/Lng'])

In [59]:
assert app_data_2022.shape[0] == original_shape - rows_removed - rows_removed_duplicate

## Add hand count data

Assign survey session ID

In [60]:
rows_removed_hand = 0
original_shape_hand = hand_count_2022.shape[0]

In [61]:
# remove column not used for anything
np.all(hand_count_2022['Name'].isna())

np.True_

In [62]:
hand_count_2022 = hand_count_2022.drop(columns = ['Name'])

In [63]:
hand_count_2022[hand_count_2022['end_datetime'] == ' ']

,counterpoint_name,lat,lng,geocoded_city,province,survey_session_id,start_datetime,end_datetime,date,gender_man,...,race_indigenous,race_latin,race_middle_eastern,race_south_asian,race_white,race_other_unsure,gender,age,race,counterpoint_id
486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3738.0


In [64]:
# delete row with nan values
hand_count_2022 = hand_count_2022[~(hand_count_2022['end_datetime'] == ' ')]

In [65]:
rows_removed_hand = rows_removed_hand + original_shape_hand - hand_count_2022.shape[0]

In [66]:
# create columns to match app data
hand_count_2022['elapsed_time_seconds'] = (pd.to_datetime(hand_count_2022['end_datetime']) 
                                         - pd.to_datetime(hand_count_2022['start_datetime'])).dt.seconds

hand_count_2022['start_datetime_hourofday'] = pd.to_datetime(hand_count_2022['start_datetime']).dt.hour
hand_count_2022['start_datetime_dayofmonth'] = pd.to_datetime(hand_count_2022['start_datetime']).dt.day
hand_count_2022['start_datetime_dayofweek'] = pd.to_datetime(hand_count_2022['start_datetime']).dt.day_name()

hand_count_2022['error'] = np.nan
hand_count_2022['team_name'] = 'Pedal Poll Canada'

In [67]:
hand_count_2022 = hand_count_2022.rename(columns = {'gender_other_unsure':'gender_unsure',
                                                   'province': 'geocoded_province'})

In [68]:
# assign bike total column
hand_count_2022['easy_bike_total'] = 1

In [69]:
# check that columns match
assert np.all(np.sort(hand_count_2022.columns) == np.sort(app_data_2022.columns))

Clean up the demographic columns for hand count data

In [70]:
hand_count_2022['age'].unique()

array(['adult', 'adult ', 'senior', 'Child', 'child', 'adul', 'senior ',
       'child ', 'other', 'adlt'], dtype=object)

In [71]:
age_dict = {'adult':'Adult', 'adult ':'Adult', 'senior':'Senior', 'Child':'Child', 'child':'Child', 'adul':'Adult', 'senior ':'Senior',
       'child ':'Child', 'other':'Other or unsure', 'adlt':'Adult'}

In [72]:
hand_count_2022['gender'].unique()

array(['man', 'woman', 'man ', 'other', 'woman ', 'other ', 'male'],
      dtype=object)

In [73]:
gender_dict = {'man':'Man', 'woman':'Woman', 'man ':'Man', 'other':'Other or unsure', 'woman ':'Woman', 'other ':'Other or unsure', 'male':'Man'}

In [74]:
hand_count_2022['race'].unique()

array(['south asian', 'black', 'white', 'other', 'White', 'east asian',
       'latin', 'middle eastern', 'indigenous', 'white ', 'whitw', 'oher'],
      dtype=object)

In [75]:
race_dict = {'south asian':'South Asian', 'black':'Black', 'other':'Other or unsure', 'white':'White', 'White':'White', 'east asian':'East Asian',
       'latin':'Latin', 'middle eastern':'Middle Eastern', 'indigenous':'Indigenous', 'white ':'White', 'whitw':'White', 'oher':'Other or unsure'}

In [76]:
hand_count_2022['age'] = hand_count_2022['age'].map(age_dict)
hand_count_2022['gender'] = hand_count_2022['gender'].map(gender_dict)
hand_count_2022['race'] = hand_count_2022['race'].map(race_dict)

In [77]:
hand_count_2022[['counterpoint_name', 'lat', 'lng', 'geocoded_city', 'geocoded_province',
       'survey_session_id', 'start_datetime', 'end_datetime', 'date']].drop_duplicates().shape

(21, 9)

In [78]:
# this uniquely identifies a session
hand_count_2022[['start_datetime', 'end_datetime', 'counterpoint_name']].drop_duplicates().shape

(21, 3)

In [79]:
session_id = 1
for i, row in hand_count_2022[['start_datetime', 'end_datetime', 'counterpoint_name']].drop_duplicates().iterrows():
    hand_count_2022.loc[(hand_count_2022['start_datetime'] == row['start_datetime'])
                    & (hand_count_2022['end_datetime'] == row['end_datetime'])
                    & (hand_count_2022['counterpoint_name'] == row['counterpoint_name']), 'survey_session_id'] = session_id
    session_id += 1

### Check for duplicated hand count and app data

In [80]:
app_data_agg = app_data_2022.groupby(['counterpoint_id', 'counterpoint_name', 'survey_session_id', 'lat', 
                         'lng', 'geocoded_city', 'start_datetime', 'start_datetime_hourofday', 
                                          'elapsed_time_seconds'])['easy_bike_total'].sum().reset_index()

In [81]:
hand_count_agg = hand_count_2022.groupby(['counterpoint_id', 'counterpoint_name', 'survey_session_id', 'geocoded_city', 'start_datetime', 
                                          'start_datetime_hourofday', 'end_datetime',
                                          'elapsed_time_seconds'])['easy_bike_total'].sum().reset_index()

In [82]:
# shared counterpoint ids between hand and app data
possible_duplicates = hand_count_agg[['survey_session_id', 'geocoded_city', 'counterpoint_id',
      'easy_bike_total', 'start_datetime', 'end_datetime']].merge(app_data_agg[['survey_session_id', 'counterpoint_id', 
                                                                    'counterpoint_name', 'start_datetime',
                                                    'easy_bike_total',]].drop_duplicates(), on = 'counterpoint_id', suffixes = ['_hand', '_app'])

In [83]:
possible_duplicates['hand_datetime'] = pd.to_datetime(possible_duplicates['start_datetime_hand'])
possible_duplicates['app_datetime'] = pd.to_datetime(possible_duplicates['start_datetime_app'])

In [84]:
# narrow it down to shared day
possible_duplicates = possible_duplicates[possible_duplicates['hand_datetime'].dt.day == possible_duplicates['app_datetime'].dt.day]

In [85]:
# narrow it down to starting time within an hour of each other
possible_duplicates = possible_duplicates[np.abs((possible_duplicates['hand_datetime'] - possible_duplicates['app_datetime']).dt.total_seconds()) < 3600]

In [86]:
possible_duplicates

,survey_session_id_hand,geocoded_city,counterpoint_id,easy_bike_total_hand,start_datetime_hand,end_datetime,survey_session_id_app,counterpoint_name,start_datetime_app,easy_bike_total_app,hand_datetime,app_datetime
5,2.0,Brampton,6387.0,15,06/08/2022 11:00,06/08/2022 13:00,3451,Chinguacousy Recreational Trail,2022-06-08 11:32,2,2022-06-08 11:00:00,2022-06-08 11:32:00
6,2.0,Brampton,6387.0,15,06/08/2022 11:00,06/08/2022 13:00,3491,Chinguacousy Recreational Trail,2022-06-08 11:50,10,2022-06-08 11:00:00,2022-06-08 11:50:00


In [87]:
# this looks like a genuine duplicate - remove the app data

In [88]:
rows_removed += app_data_2022[(app_data_2022['survey_session_id'].isin(possible_duplicates['survey_session_id_app']))].shape[0]

In [89]:
app_data_2022 = app_data_2022[~(app_data_2022['survey_session_id'].isin(possible_duplicates['survey_session_id_app']))]

In [90]:
# save data
all_data = pd.concat([app_data_2022, hand_count_2022])

In [91]:
all_data = all_data.drop(columns = 'easy_bike_total')

In [92]:
rows_removed += all_data[all_data['counterpoint_id'].isna()].shape[0]
all_data = all_data[~all_data['counterpoint_id'].isna()]

In [93]:
all_data.to_csv("data/2022_cleaned_data.csv", index = None)

In [94]:
# check that we only removed the rows we meant to
assert all_data.shape[0] == original_shape + original_shape_hand - rows_removed - rows_removed_hand - rows_removed_duplicate

In [95]:
np.sum(np.any(hand_count_2022[['survey_session_id','counterpoint_id']].drop_duplicates().isna(), axis = 1))

np.int64(9)